In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
raw_data = project_root / "data" / "raw" / "customer_support_tickets.csv"
db = project_root / "data" / "processed" / "customer_support.db"

In [2]:
q11 = '''
WITH type_ratings AS(
    SELECT 
        tt.ticket_type_name, 
        AVG(t.customer_satisfaction_rating) as avg_rating
    FROM tickets t
    JOIN ticket_types tt ON tt.ticket_type_id = t.ticket_type_id
    GROUP BY ticket_type_name
)
SELECT *
FROM type_ratings
WHERE avg_rating < 3
ORDER BY avg_rating;
'''
with sqlite3.connect(db) as conn:
    result11 = pd.read_sql(q11, conn)

result11

,ticket_type_name,avg_rating
0,Refund request,2.934564
1,Technical issue,2.958621


In [3]:
create_view_q = '''
CREATE VIEW IF NOT EXISTS critical_tickets_view AS
SELECT 
    t.ticket_id,
    c.customer_name,
    p.product_name,
    t.ticket_channel,
    t.customer_satisfaction_rating
FROM tickets t
JOIN customers c ON c.customer_id = t.customer_id
JOIN products p ON p.product_id = t.product_id
WHERE t.ticket_priority = 'Critical';
'''

check_q = '''
SELECT * FROM critical_tickets_view
LIMIT 5;
'''

with sqlite3.connect(db) as conn:
    conn.execute(create_view_q)
    result12 = pd.read_sql(check_q, conn)

result12

,ticket_id,customer_name,product_name,ticket_channel,customer_satisfaction_rating
0,1,Marisa Obrien,GoPro Hero,Social media,None
1,2,Jessica Rios,LG Smart TV,Chat,None
2,7,Jacqueline Wright,Microsoft Surface,Social media,None
3,8,Denise Lee,Philips Hue Lights,Social media,None
4,10,William Dawson,Dyson Vacuum Cleaner,Phone,None
